In [1]:
import pandas as pd
from hitsUtils import HitsProcessor

In [2]:
df = pd.read_csv('/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirClassification/linnaeus_hits/sorted.hits', sep='\t', quoting=3, names=HitsProcessor._COL_NAMES)
df.head()

,sentence_id,synonym_id,matched_text,start_position,hit_length,synonym,prefix,suffix
0,chunk_00.sent:PMC350664.2.3,0:5363,HeLa cells,172,10,HeLa cells,NaN,.
1,chunk_00.sent:PMC350664.2.3,0:5363,HeLa cell,126,9,HeLa cell,NaN,NaN
2,chunk_00.sent:PMC350664.2.3,1:1359,Drosophila,96,10,Drosophila,NaN,NaN
3,chunk_00.sent:PMC350664.2.3,2:261994,human,166,5,Human,NaN,NaN
4,chunk_00.sent:PMC350664.2.4,2:586693,"Caenorhabditis elegans,",3,23,Caenorhabditis elegans,NaN,NaN


In [3]:
df['sentence_id'] = df['sentence_id'].str.split(':').str[1]
df.head()

,sentence_id,synonym_id,matched_text,start_position,hit_length,synonym,prefix,suffix
0,PMC350664.2.3,0:5363,HeLa cells,172,10,HeLa cells,NaN,.
1,PMC350664.2.3,0:5363,HeLa cell,126,9,HeLa cell,NaN,NaN
2,PMC350664.2.3,1:1359,Drosophila,96,10,Drosophila,NaN,NaN
3,PMC350664.2.3,2:261994,human,166,5,Human,NaN,NaN
4,PMC350664.2.4,2:586693,"Caenorhabditis elegans,",3,23,Caenorhabditis elegans,NaN,NaN


In [ ]:
# 1. Calculate the end boundary
df["end_position"] = df["start_position"] + df["hit_length"]

# 2. Sort by start_position, preserving the sentence groups
# (Using kind='stable' keeps the existing sentence order perfectly intact)
df = df.sort_values(
    by=["sentence_id", "start_position"], kind="stable"
).reset_index(drop=True)

# 3. Compute rolling max end position within each sentence group
df["max_end_so_far"] = (
    df.groupby("sentence_id")["end_position"].shift(1).cummax()
)

# 4. Identify overlaps
df["is_overlapping"] = df["start_position"] <= df["max_end_so_far"]

# 5. Extract rows that overlap with a previous entity in the same sentence
overlapping_hits = df[df["is_overlapping"] == True]

AttributeError: 'OutStream' object has no attribute 'buffer'